1. Setup and Importing Packages

In [1]:
import pandas as pd 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score

RANDOM_STATE = 1213

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "retailhero_causal_forest_features.csv")

df = pd.read_csv(PROCESSED_DATA_PATH)

print(df.shape)
display(df.head())
print(df.columns.tolist())

(200039, 25)


,client_id,age,gender_F,gender_M,gender_U,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,...,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span,treatment,target
0,000012768d,45,0,0,1,4,46,3,54.0,13.500000,...,0.0,0.0,0.0,25.7,0.0,4,34.441906,103,0,1
1,000036f903,72,1,0,0,32,96,5,169.0,5.281250,...,0.0,60.0,0.0,54.9,60.0,1,3.515704,108,1,1
2,00010925a5,83,0,0,1,18,58,2,79.0,4.388889,...,-17.0,0.0,0.0,14.8,0.0,10,6.049572,102,1,1
3,0001f552b0,33,1,0,0,15,79,4,106.0,7.066667,...,0.0,0.0,0.0,78.9,0.0,2,8.010879,112,1,1
4,00020e7b18,73,0,0,1,18,175,4,394.0,21.888889,...,-592.0,0.0,-30.0,-305.9,-30.0,3,6.597343,112,1,1


['client_id', 'age', 'gender_F', 'gender_M', 'gender_U', 'num_transactions', 'num_unique_products', 'num_stores_visited', 'total_quantity', 'avg_items_per_transaction', 'total_spend', 'avg_transaction_spend', 'median_transaction_spend', 'spend_std', 'regular_points_received', 'regular_points_spent', 'express_points_received', 'express_points_spent', 'net_regular_points_change', 'net_express_points_change', 'days_since_last_purchase', 'avg_days_between_transactions', 'customer_activity_span', 'treatment', 'target']


2. Definiting X, Y and D

In [2]:
D = df["treatment"] #marketing treatment
Y = df["target"] #purchase outcome

X = df.drop(columns=["client_id", "treatment", "target","gender_U"])  #customer characteristics pre-treatment

print("X:", X.shape)
print("D:", D.shape)
print("Y:", Y.shape)

# Sanity check: make sure that the treatment and outcome are binary
display(X.head())
print("Treatment distribution:")
display(D.value_counts(normalize=True).rename("proportion"))

print("Outcome distribution:")
display(Y.value_counts(normalize=True).rename("proportion"))


X: (200039, 21)
D: (200039,)
Y: (200039,)


,age,gender_F,gender_M,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,total_spend,avg_transaction_spend,...,spend_std,regular_points_received,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span
0,45,0,0,4,46,3,54.0,13.500000,2803.00,700.750000,...,258.000484,25.7,0.0,0.0,0.0,25.7,0.0,4,34.441906,103
1,72,1,0,32,96,5,169.0,5.281250,9805.00,306.406250,...,161.780987,54.9,0.0,60.0,0.0,54.9,60.0,1,3.515704,108
2,83,0,0,18,58,2,79.0,4.388889,5883.00,326.833333,...,139.562868,31.8,-17.0,0.0,0.0,14.8,0.0,10,6.049572,102
3,33,1,0,15,79,4,106.0,7.066667,6155.18,410.345333,...,295.885838,78.9,0.0,0.0,0.0,78.9,0.0,2,8.010879,112
4,73,0,0,18,175,4,394.0,21.888889,25819.61,1434.422778,...,1008.952394,286.1,-592.0,0.0,-30.0,-305.9,-30.0,3,6.597343,112


Treatment distribution:


treatment
0    0.500192
1    0.499808
Name: proportion, dtype: float64

Outcome distribution:


target
1    0.619889
0    0.380111
Name: proportion, dtype: float64

4. Standardisation

In [3]:
feature_info = pd.DataFrame({
    "dtype": X.dtypes,
    "n_unique": X.nunique(),
    "min": X.min(),
    "max": X.max()
})

#display(feature_info)

binary_cols = [
    col for col in X.columns
    if set(X[col].dropna().unique()).issubset({0, 1})
]

# identify continuous and binary columns

continuous_cols = [
    col for col in X.columns
    if col not in binary_cols
]

print("Binary variables:")
print(binary_cols)

print("\nContinuous/count variables:")
print(continuous_cols)

# stadardise continuous variables

scaler = StandardScaler()

x_scaled = X.copy()

x_scaled[continuous_cols] = scaler.fit_transform(x_scaled[continuous_cols]) 

# sanity check for normalisation

display(x_scaled[continuous_cols].describe().loc[["mean", "std"]])

Binary variables:
['gender_F', 'gender_M']

Continuous/count variables:
['age', 'num_transactions', 'num_unique_products', 'num_stores_visited', 'total_quantity', 'avg_items_per_transaction', 'total_spend', 'avg_transaction_spend', 'median_transaction_spend', 'spend_std', 'regular_points_received', 'regular_points_spent', 'express_points_received', 'express_points_spent', 'net_regular_points_change', 'net_express_points_change', 'days_since_last_purchase', 'avg_days_between_transactions', 'customer_activity_span']


,age,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,total_spend,avg_transaction_spend,median_transaction_spend,spend_std,regular_points_received,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span
mean,1.286720e-16,2.312366e-17,-1.236103e-17,-1.080880e-16,7.942319e-17,-1.059213e-16,2.482863e-17,1.395944e-17,-2.484284e-16,2.181651e-16,4.031544e-17,-5.185951e-18,-2.291942e-17,-7.796686e-18,-5.860835e-18,3.179947e-17,-3.306932e-17,-1.045715e-16,1.571592e-16
std,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00,1.000002e+00


5. Construction of treatment interactions


In [4]:
x_interactions = x_scaled.mul(D, axis=0) 

x_interactions.columns = [f"{col}_treatment" for col in x_interactions.columns]

display(x_interactions.head())

d_df = D.rename("treatment").to_frame()

Z = pd.concat([d_df, x_scaled, x_interactions], axis=1)


print("design matrix shape:", Z.shape)

display(Z.head())

,age_treatment,gender_F_treatment,gender_M_treatment,num_transactions_treatment,num_unique_products_treatment,num_stores_visited_treatment,total_quantity_treatment,avg_items_per_transaction_treatment,total_spend_treatment,avg_transaction_spend_treatment,...,spend_std_treatment,regular_points_received_treatment,regular_points_spent_treatment,express_points_received_treatment,express_points_spent_treatment,net_regular_points_change_treatment,net_express_points_change_treatment,days_since_last_purchase_treatment,avg_days_between_transactions_treatment,customer_activity_span_treatment
0,-0.000000,0,0,-0.000000,-0.000000,0.000000,-0.000000,0.000000,-0.000000,0.000000,...,-0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000
1,1.611625,1,0,0.669926,0.367386,1.036953,0.188345,-0.502968,0.135450,-0.505615,...,-0.598193,-0.234555,0.553872,7.733373,0.395826,0.560578,3.793654,-0.851949,-0.538801,0.600922
2,2.303027,0,0,-0.119604,-0.306628,-0.459149,-0.451995,-0.667564,-0.304504,-0.451077,...,-0.682532,-0.468877,0.425586,-0.104071,0.395826,0.113447,0.326893,0.735910,-0.296668,0.389458
3,-0.839707,1,0,-0.288788,0.065854,0.538252,-0.259893,-0.173646,-0.273972,-0.228110,...,-0.089137,0.008897,0.553872,-0.104071,0.395826,0.828189,0.326893,-0.675520,-0.109248,0.741898
4,1.674480,0,0,-0.119604,1.768626,0.538252,1.789195,2.560327,1.931902,2.506062,...,2.617634,2.110701,-3.913501,-0.104071,-1.443988,-3.462493,-1.406487,-0.499091,-0.244323,0.741898


design matrix shape: (200039, 43)


,treatment,age,gender_F,gender_M,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,total_spend,...,spend_std_treatment,regular_points_received_treatment,regular_points_spent_treatment,express_points_received_treatment,express_points_spent_treatment,net_regular_points_change_treatment,net_express_points_change_treatment,days_since_last_purchase_treatment,avg_days_between_transactions_treatment,customer_activity_span_treatment
0,0,-0.085451,0,0,-0.909133,-0.519474,0.039552,-0.629867,1.012989,-0.650005,...,-0.000000,-0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000
1,1,1.611625,1,0,0.669926,0.367386,1.036953,0.188345,-0.502968,0.135450,...,-0.598193,-0.234555,0.553872,7.733373,0.395826,0.560578,3.793654,-0.851949,-0.538801,0.600922
2,1,2.303027,0,0,-0.119604,-0.306628,-0.459149,-0.451995,-0.667564,-0.304504,...,-0.682532,-0.468877,0.425586,-0.104071,0.395826,0.113447,0.326893,0.735910,-0.296668,0.389458
3,1,-0.839707,1,0,-0.288788,0.065854,0.538252,-0.259893,-0.173646,-0.273972,...,-0.089137,0.008897,0.553872,-0.104071,0.395826,0.828189,0.326893,-0.675520,-0.109248,0.741898
4,1,1.674480,0,0,-0.119604,1.768626,0.538252,1.789195,2.560327,1.931902,...,2.617634,2.110701,-3.913501,-0.104071,-1.443988,-3.462493,-1.406487,-0.499091,-0.244323,0.741898


6. Cross Validation + Lasso

In [5]:
# 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Candidate C values
# Smaller C = stronger regularisation
c_grid = np.logspace(-3, 2, 20)

# Fit cross-validated LASSO logistic regression
lasso_cv = LogisticRegressionCV(
    Cs=c_grid,
    cv=cv,
    penalty="l1",
    solver="saga",
    scoring="neg_log_loss",
    random_state=RANDOM_STATE,
    refit=True,
    max_iter=5000,
    n_jobs=-1
)

lasso_cv.fit(Z,Y)

# ---------------------------------------------------------
# Apply the 1-standard-error (1-SE) rule
# ---------------------------------------------------------

score_key = list(lasso_cv.scores_.keys())[0]
cv_scores = lasso_cv.scores_[score_key]

# Mean CV score and standard error for each C
mean_scores = cv_scores.mean(axis=0)
se_scores = cv_scores.std(axis=0, ddof=1) / np.sqrt(cv_scores.shape[0])

# Best-performing C
best_idx = np.argmax(mean_scores)
best_c = lasso_cv.Cs_[best_idx]
best_score = mean_scores[best_idx]
best_se = se_scores[best_idx]

# 1-SE threshold
one_se_threshold = best_score - best_se

# Find all C values whose performance is within 1 SE of the best
eligible_idx = np.where(mean_scores >= one_se_threshold)[0]

# Choose smallest C = strongest regularisation within 1 SE
one_se_c = lasso_cv.Cs_[eligible_idx[0]]

print(f"CV-optimal C: {best_c:.6f}")
print(f"1-SE C:       {one_se_c:.6f}")

# ---------------------------------------------------------
# Refit LASSO using the more parsimonious 1-SE C
# ---------------------------------------------------------

lasso_1se = LogisticRegression(
    penalty="l1",
    solver="saga",
    C=one_se_c,
    random_state=RANDOM_STATE,
    max_iter=5000
)

lasso_1se.fit(Z,Y)

CV-optimal C: 0.069519
1-SE C:       0.003360


,penalty,'l1'
,dual,False
,tol,0.0001
,C,np.float64(0....9818286283781)
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,1213
,solver,'saga'
,max_iter,5000
,multi_class,'deprecated'


In [6]:
# Extract coefficients from the 1-SE LASSO model
coef_df = pd.DataFrame({
    "variable": Z.columns,
    "coefficient": lasso_1se.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    by="abs_coefficient",
    ascending=False
)

# Keep only treatment interaction terms
interaction_results = coef_df[
    coef_df["variable"].str.endswith("_treatment")
].copy()

# LASSO-selected interactions have non-zero coefficients
interaction_results["selected"] = (
    interaction_results["abs_coefficient"] > 1e-8
)

interaction_results = interaction_results.sort_values(
    by="abs_coefficient",
    ascending=False
)

print("All treatment interactions:")
display(interaction_results)

# Keep only selected interactions
selected_interactions = interaction_results[
    interaction_results["selected"]
].copy()

print(
    f"\n{len(selected_interactions)} out of "
    f"{len(interaction_results)} treatment interactions selected "
    f"using the 1-SE rule."
)

print("\nSelected treatment interactions:")
display(
    selected_interactions[
        ["variable", "coefficient", "abs_coefficient"]
    ]
)

All treatment interactions:


,variable,coefficient,abs_coefficient,selected
37,express_points_spent_treatment,-0.057113,0.057113,True
22,age_treatment,0.039799,0.039799,True
23,gender_F_treatment,0.031910,0.031910,True
27,num_stores_visited_treatment,-0.013602,0.013602,True
33,spend_std_treatment,-0.011221,0.011221,True
26,num_unique_products_treatment,0.000000,0.000000,False
25,num_transactions_treatment,0.000000,0.000000,False
24,gender_M_treatment,0.000000,0.000000,False
29,avg_items_per_transaction_treatment,0.000000,0.000000,False
30,total_spend_treatment,0.000000,0.000000,False



5 out of 21 treatment interactions selected using the 1-SE rule.

Selected treatment interactions:


,variable,coefficient,abs_coefficient
37,express_points_spent_treatment,-0.057113,0.057113
22,age_treatment,0.039799,0.039799
23,gender_F_treatment,0.031910,0.031910
27,num_stores_visited_treatment,-0.013602,0.013602
33,spend_std_treatment,-0.011221,0.011221


7. Save Lasso-selected features for output downstream

In [7]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

selected_features = (selected_interactions["variable"].str.replace("_treatment", "", regex=False).tolist())

selected_output = pd.DataFrame({"feature": selected_features})

OUTPUT_PATH = OUTPUT_DIR / "lasso_selected_features.csv"

selected_output.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(selected_features)} LASSO-selected features:")
for feature in selected_features:
    print(f"  - {feature}")

print(f"\nSaved to: {OUTPUT_PATH.resolve()}")

Saved 5 LASSO-selected features:
  - express_points_spent
  - age
  - gender_F
  - num_stores_visited
  - spend_std

Saved to: C:\Users\szepi\OneDrive\Documents\dse4101\DSE4101-CausalForest-Grp2\data\output\lasso_selected_features.csv
